# Part F: Support Vector Regression (SVR)
**Robust Regression Engine**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.linear_model import Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv(r"/mnt/user-data/uploads/Advanced_Regression_HousePrice_Dataset_3800_-_Advanced_Regression_HousePrice_Dataset_3800_csv.csv")

TARGET    = 'house_price_inr'
DROP_COLS = ['property_id', 'sale_date']
FEATURES  = [col for col in df.columns if col not in [TARGET] + DROP_COLS]

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# SVR is slow on large data — use a subset for training
np.random.seed(42)
idx = np.random.choice(len(X_train_scaled), 800, replace=False)
X_sub  = X_train_scaled[idx]
y_sub  = y_train.values[idx]

print("Data Ready")
print("Subset shape for SVR:", X_sub.shape)

## Task 19 — SVR with Linear Kernel

In [ ]:
svr_linear = SVR(kernel='linear', C=1e7, epsilon=1e5)

svr_linear.fit(X_sub, y_sub)

y_pred_linear = svr_linear.predict(X_test_scaled)

print("===== SVR (Linear Kernel) =====")
print("Training Score :", svr_linear.score(X_sub, y_sub))
print("Testing Score  :", r2_score(y_test, y_pred_linear))
print("RMSE           :", np.sqrt(mean_squared_error(y_test, y_pred_linear)))
print("MAE            :", mean_absolute_error(y_test, y_pred_linear))

## Task 19 — SVR with RBF Kernel

In [ ]:
svr_rbf = SVR(kernel='rbf', C=1e7, epsilon=1e5)

svr_rbf.fit(X_sub, y_sub)

y_pred_rbf = svr_rbf.predict(X_test_scaled)

print("===== SVR (RBF Kernel) =====")
print("Training Score :", svr_rbf.score(X_sub, y_sub))
print("Testing Score  :", r2_score(y_test, y_pred_rbf))
print("RMSE           :", np.sqrt(mean_squared_error(y_test, y_pred_rbf)))
print("MAE            :", mean_absolute_error(y_test, y_pred_rbf))

## Task 19 — SVR with Polynomial Kernel

In [ ]:
svr_poly = SVR(kernel='poly', degree=2, C=1e7, epsilon=1e5)

svr_poly.fit(X_sub, y_sub)

y_pred_poly = svr_poly.predict(X_test_scaled)

print("===== SVR (Polynomial Kernel) =====")
print("Training Score :", svr_poly.score(X_sub, y_sub))
print("Testing Score  :", r2_score(y_test, y_pred_poly))
print("RMSE           :", np.sqrt(mean_squared_error(y_test, y_pred_poly)))
print("MAE            :", mean_absolute_error(y_test, y_pred_poly))

## Task 20 — Tune Hyperparameters C, epsilon, gamma

In [ ]:
param_grid = {
    'C':       [1e6, 1e7, 5e7],
    'epsilon': [5e4, 1e5, 5e5],
    'gamma':   ['scale', 'auto']
}

grid = GridSearchCV(
    SVR(kernel='rbf'),
    param_grid,
    cv=3,
    scoring='r2',
    n_jobs=-1,
    verbose=1
)

grid.fit(X_sub, y_sub)

print("Best Params :", grid.best_params_)
print("Best CV R2  :", grid.best_score_)

In [ ]:
best_svr = grid.best_estimator_

y_pred_svr = best_svr.predict(X_test_scaled)

print("===== Best SVR (RBF) =====")
print("Testing Score :", r2_score(y_test, y_pred_svr))
print("RMSE          :", np.sqrt(mean_squared_error(y_test, y_pred_svr)))
print("MAE           :", mean_absolute_error(y_test, y_pred_svr))

## Task 21 — Compare SVR vs Lasso vs Random Forest

In [ ]:
lasso = Lasso(alpha=1000, max_iter=50000)
lasso.fit(X_train_scaled, y_train)
y_pred_lasso = lasso.predict(X_test_scaled)

rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print(f"{'Model':<25} {'Test R2':>10} {'RMSE':>15} {'MAE':>13}")
print("-" * 68)

for name, yp in [
    ('SVR (best RBF)', y_pred_svr),
    ('Lasso (alpha=1000)', y_pred_lasso),
    ('Random Forest (100)', y_pred_rf),
]:
    r2   = r2_score(y_test, yp)
    rmse = np.sqrt(mean_squared_error(y_test, yp))
    mae  = mean_absolute_error(y_test, yp)
    print(f"{name:<25} {r2:>10.4f} {rmse:>15,.0f} {mae:>13,.0f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

combos = [
    ('SVR (best RBF)', y_pred_svr),
    ('Lasso', y_pred_lasso),
    ('Random Forest', y_pred_rf),
]

for ax, (name, yp) in zip(axes, combos):
    ax.scatter(y_test / 1e6, yp / 1e6, alpha=0.3, s=12)
    mn = min(y_test.min(), yp.min()) / 1e6
    mx = max(y_test.max(), yp.max()) / 1e6
    ax.plot([mn, mx], [mn, mx], 'r--', linewidth=1)
    ax.set_xlabel('Actual (M INR)')
    ax.set_ylabel('Predicted (M INR)')
    ax.set_title(f'{name}\nR2={r2_score(y_test, yp):.3f}')

plt.tight_layout()
plt.show()